# Rank Ensemble (Final Submission)

Combines predictions from TF-IDF, LSTM, and DeBERTa models using a **weighted rank ensemble**. Each model assigns a score to each option based on its rank in the top-3, then scores are summed across models with the strongest model (DeBERTa) getting the highest weight.

This is the final submission notebook — it takes individual model CSVs and produces a single blended prediction file.

### 1. Experiment Tracking (WandB)

In [1]:
import wandb
try:
    from kaggle_secrets import UserSecretsClient
    wandb_key = UserSecretsClient().get_secret('WANDB_API_KEY')
    wandb.login(key=wandb_key)
    wandb.init(
        project='23f2003236-t22026',
        name='ensemble-4model',
        config={
            'tfidf_weight': 0.15,
            'lstm_weight': 0.20,
            'deberta_weight': 0.40,
            'roberta_weight': 0.25,
            'method': 'rank_ensemble',
            'models': ['tfidf', 'lstm', 'deberta', 'roberta_mc']
        }
    )
    print('WandB connected')
except Exception:
    print('WandB not available, continuing offline')
    USE_WANDB = False

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 23f2003236 (23f2003236-iit-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260714_164238-f4vyef8q
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run ensemble-4model
wandb: ⭐️ View project at https://wandb.ai/23f2003236-iit-madras/23f2003236-t22026
wandb: 🚀 View run at https://wandb.ai/23f2003236-iit-madras/23f2003236-t22026/runs/f4vyef8q


WandB connected


### 2. Load Individual Model Submissions

Each CSV has `ID` and `Prediction` columns where `Prediction` is a space-separated string of 3 answer letters (e.g. `"B D A"`).

In [2]:
import pandas as pd
import numpy as np
from collections import defaultdict

# file paths to individual model submissions
path_tfidf   = '/kaggle/input/datasets/lukerohan/best-csv/submission_tfidf.csv'
path_lstm    = '/kaggle/input/datasets/lukerohan/best-csv/submission_lstm.csv'
path_deberta = '/kaggle/input/datasets/lukerohan/best-csv/submission_DeBERTa.csv'
path_roberta = '/kaggle/input/datasets/lukerohan/best-csv/submission_roberta_mc.csv'

sub_tfidf   = pd.read_csv(path_tfidf)
sub_lstm    = pd.read_csv(path_lstm)
sub_deberta = pd.read_csv(path_deberta)
sub_roberta = pd.read_csv(path_roberta)

print(f'TF-IDF:  {sub_tfidf.shape}')
print(f'LSTM:    {sub_lstm.shape}')
print(f'DeBERTa: {sub_deberta.shape}')
print(f'RoBERTa: {sub_roberta.shape}')

# verify the same number of rows across all four files
assert len(sub_tfidf) == len(sub_lstm) == len(sub_deberta) == len(sub_roberta), 'Row count mismatch!'

TF-IDF:  (500, 2)
LSTM:    (500, 2)
DeBERTa: (500, 2)
RoBERTa: (500, 2)


### 3. Weighted Rank Ensemble — how it actually works

Each model doesn't just output one answer — it outputs its **top-3 guesses in order**, since that's what MAP@3 rewards. The idea behind this ensemble is simple: instead of trusting one model's ranking blindly, let all four models "vote," but let the stronger models' votes count for more.

For every question, and for every model, we go through that model's top-3 predictions and give each option a score based on where it was ranked:

- The model's **1st choice** gets the model's full weight (`weight × 1.0`)
- The model's **2nd choice** gets half the weight (`weight × 0.5`)
- The model's **3rd choice** gets a third of the weight (`weight × 0.333`)

This `1 / (rank + 1)` pattern is called **reciprocal rank scoring** — it's a standard way to combine ranked lists from different sources. It's deliberately not linear (rank 1 isn't just "a bit better" than rank 2, it's worth twice as much), because MAP@3 itself scores a rank-1 hit as 1.0 and a rank-2 hit as only 0.5 — so the ensemble's internal scoring mirrors the actual competition metric.

These per-model, per-option scores are then **summed across all four models**. If two or three models independently agree that option `B` is a strong candidate, `B`'s combined score stacks up fast, even if no single model was fully confident on its own. Whichever three options end up with the highest combined score become the final top-3 prediction, in order.

**Why DeBERTa gets the largest weight (0.40) and TF-IDF the smallest (0.15):** the weights roughly reflect each model's standalone leaderboard performance — DeBERTa is the strongest individual model, so its vote should count for more, while TF-IDF is the weakest individually but still contributes something the other three occasionally miss, since it's the only non-neural model in the mix and makes different kinds of mistakes. LSTM and RoBERTa sit in between. These exact weights (`0.15 / 0.20 / 0.40 / 0.25`) were reached by testing a few combinations directly against the leaderboard, since with a ~500-row test set the difference between nearby weight choices is small enough that it has to be verified empirically rather than assumed.

In [3]:
# ensemble weights — tuned by testing a few combinations on the leaderboard
W_TFIDF   = 0.15
W_LSTM    = 0.20
W_DEBERTA = 0.40
W_ROBERTA = 0.25

# sanity check: weights should add up to 1.0
assert abs((W_TFIDF + W_LSTM + W_DEBERTA + W_ROBERTA) - 1.0) < 1e-9, 'Weights must sum to 1.0!'

final_predictions = []

for idx in range(len(sub_tfidf)):
    scores = defaultdict(float)

    # score TF-IDF predictions
    preds = str(sub_tfidf.iloc[idx]['Prediction']).split()
    for rank, pred in enumerate(preds):
        scores[pred] += W_TFIDF * (1.0 / (rank + 1))

    # score LSTM predictions
    preds = str(sub_lstm.iloc[idx]['Prediction']).split()
    for rank, pred in enumerate(preds):
        scores[pred] += W_LSTM * (1.0 / (rank + 1))

    # score DeBERTa predictions
    preds = str(sub_deberta.iloc[idx]['Prediction']).split()
    for rank, pred in enumerate(preds):
        scores[pred] += W_DEBERTA * (1.0 / (rank + 1))

    # score RoBERTa-MC predictions
    preds = str(sub_roberta.iloc[idx]['Prediction']).split()
    for rank, pred in enumerate(preds):
        scores[pred] += W_ROBERTA * (1.0 / (rank + 1))

    # take top-3 options by combined score
    best_3 = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:3]
    final_predictions.append(' '.join([x[0] for x in best_3]))

print(f'Ensemble complete. {len(final_predictions)} predictions generated.')

Ensemble complete. 500 predictions generated.


In [4]:
format_errors = sum(
    1 for p in final_predictions
    if len(p.split()) != 3 or not all(c in 'ABCDE' for c in p.split())
)
print(f'Format errors: {format_errors} (must be 0 before submitting!)')

Format errors: 0 (must be 0 before submitting!)


### 4. Save & Log

In [5]:
# build submission dataframe
id_col = sub_tfidf.columns[0]
final_submission = pd.DataFrame({
    id_col: sub_tfidf[id_col],
    'Prediction': final_predictions
})

# save
save_path = '/kaggle/working/ensemble_4model.csv'
final_submission.to_csv(save_path, index=False)

# log artifact to WandB
try:
    artifact = wandb.Artifact(
        name='final-ensemble-submission-4model',
        type='submission'
    )
    artifact.add_file(save_path)
    wandb.log_artifact(artifact)
except Exception:
    pass

print(f'Saved: {save_path}')
print(final_submission.head())

try:
    wandb.finish()
except Exception:
    pass

wandb: updating run metadata; uploading artifact final-ensemble-submission-4model


Saved: /kaggle/working/ensemble_4model.csv
   ID Prediction
0   1      A B E
1   2      B D E
2   3      B A E
3   4      E C D
4   5      C A D


wandb: uploading wandb-metadata.json; uploading requirements.txt; uploading output.log; uploading wandb-summary.json; uploading config.yaml
wandb: 🚀 View run ensemble-4model at: https://wandb.ai/23f2003236-iit-madras/23f2003236-t22026/runs/f4vyef8q
wandb: ⭐️ View project at: https://wandb.ai/23f2003236-iit-madras/23f2003236-t22026
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260714_164238-f4vyef8q/logs
wandb: WARNING Artifact "final-ensemble-submission-4model" already exists with the same content. No new version will be created.
